# LLM Auction Experiment — Data Analysis

This notebook reads `results.csv` produced by `main.py` and visualizes:
1. Bid vs Value scatter plot with BNE and 45° reference lines
2. Convergence to BNE over rounds
3. Bid ratio (bid/value) over time
4. Cumulative profit per agent
5. Win rate distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

RESULTS_FILE = "results.csv"
N_AGENTS = 4       # must match api.env / main.py setting
VALUE_LOW = 50.0   # must match api.env VALUE_LOW
VALUE_HIGH = 80.0  # must match api.env VALUE_HIGH

def bne(v):
    """Symmetric BNE bid in a first-price auction, values iid U(VALUE_LOW, VALUE_HIGH)."""
    return VALUE_LOW + (N_AGENTS - 1) / N_AGENTS * (v - VALUE_LOW)


df = pd.read_csv(RESULTS_FILE)
# Recompute the benchmark from values, so CSVs written by older code are also handled correctly
df["bne_bid"] = bne(df["value"])
df["bne_ratio"] = df["bne_bid"] / df["value"]
print(f"Loaded {len(df)} rows across {df['round'].nunique()} rounds and {df['agent_id'].nunique()} agents.")
df.head(10)

## 1. Bid vs Value — Full Experiment

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Color points by round number to show temporal evolution
scatter = ax.scatter(
    df["value"], df["bid"],
    c=df["round"], cmap="viridis", alpha=0.6, s=30, label="Agent bids"
)
plt.colorbar(scatter, ax=ax, label="Round")

v_range = np.linspace(VALUE_LOW, VALUE_HIGH, 200)

# 45° honest bidding line
ax.plot(v_range, v_range, "r--", lw=1.5, label="45° (bid = value)")

# BNE line: b* = L + (N-1)/N * (v - L)
ax.plot(v_range, bne(v_range), "g-", lw=2,
        label=f"BNE (N={N_AGENTS}, v~U({VALUE_LOW:g},{VALUE_HIGH:g}))")

ax.set_xlabel("Private Value", fontsize=12)
ax.set_ylabel("Bid", fontsize=12)
ax.set_title("Bid vs Value — All Rounds", fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig("plot_bid_vs_value.png", dpi=150)
plt.show()

## 2. Convergence to BNE — Early vs Late Rounds

In [ ]:
max_round = df["round"].max()
cutoff = max_round // 2

early = df[df["round"] <= cutoff]
late  = df[df["round"] >  cutoff]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, subset, title in zip(axes, [early, late], [f"Early Rounds (1–{cutoff})", f"Late Rounds ({cutoff+1}–{max_round})"]):
    ax.scatter(subset["value"], subset["bid"], alpha=0.5, s=25, color="steelblue")
    ax.plot(v_range, v_range, "r--", lw=1.5, label="45° line")
    ax.plot(v_range, bne(v_range), "g-", lw=2, label="BNE")
    ax.set_xlabel("Private Value")
    ax.set_ylabel("Bid")
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle("Convergence to BNE", fontsize=14)
plt.tight_layout()
plt.savefig("plot_convergence.png", dpi=150)
plt.show()

## 3. Bid Ratio (bid / value) Over Rounds

With values on U(L, H), the BNE bid ratio b*(v)/v depends on v, so the benchmark is the round average of b*(v)/v (it equals (N-1)/N only when L = 0).

In [ ]:
avg_ratio = df.groupby("round")["bid_ratio"].mean()
rolling_avg = avg_ratio.rolling(window=5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(avg_ratio.index, avg_ratio.values, alpha=0.4, s=20, label="Round avg")
ax.plot(rolling_avg.index, rolling_avg.values, "b-", lw=2, label="5-round rolling avg")
bne_ratio_round = df.groupby("round")["bne_ratio"].mean()
ax.plot(bne_ratio_round.index, bne_ratio_round.values, color="green", lw=1.5,
        label="BNE target (round avg)")
ax.axhline(1.0, color="red", linestyle="--", lw=1, label="Honest bidding (ratio=1)")
ax.set_xlabel("Round")
ax.set_ylabel("Bid / Value")
ax.set_title("Bid Ratio Over Time", fontsize=13)
ax.set_ylim(0, 1.2)
ax.legend()
plt.tight_layout()
plt.savefig("plot_bid_ratio.png", dpi=150)
plt.show()

## 4. Cumulative Profit Per Agent

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = cm.tab10.colors

for agent_id, group in df.groupby("agent_id"):
    group = group.sort_values("round")
    persona = group["persona"].iloc[0]
    ax.plot(
        group["round"], group["cumulative_profit"],
        lw=2, color=colors[agent_id],
        label=f"Agent {agent_id} ({persona})"
    )

ax.set_xlabel("Round")
ax.set_ylabel("Cumulative Profit")
ax.set_title("Cumulative Profit Per Agent", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("plot_cumulative_profit.png", dpi=150)
plt.show()

## 5. Summary Statistics

In [ ]:
summary = df.groupby(["agent_id", "persona"]).agg(
    avg_bid_ratio=("bid_ratio", "mean"),
    std_bid_ratio=("bid_ratio", "std"),
    win_rate=("won", "mean"),
    total_profit=("payoff", "sum"),
    avg_profit_per_round=("payoff", "mean"),
).round(4)

summary["bne_target"] = df.groupby(["agent_id", "persona"])["bne_ratio"].mean().round(4)
summary["ratio_deviation"] = (summary["avg_bid_ratio"] - summary["bne_target"]).abs()
# Deviation in bid levels, which is comparable across value ranges
summary["mean_abs_bid_gap"] = (df["bid"] - df["bne_bid"]).abs().groupby([df["agent_id"], df["persona"]]).mean().round(4)

print(summary.to_string())
summary

In [ ]:
# Late-round average bid ratio vs BNE benchmark
late_df = df[df["round"] > cutoff]
late_ratio = late_df["bid_ratio"].mean()
late_target = late_df["bne_ratio"].mean()
print(f"Late-round average bid ratio : {late_ratio:.4f}")
print(f"BNE theoretical target       : {late_target:.4f}")
print(f"Deviation                    : {abs(late_ratio - late_target):.4f}")
print(f"Late-round mean |bid - BNE|  : {(late_df['bid'] - late_df['bne_bid']).abs().mean():.4f}")